In [7]:
import pandas as pd
import time
import sys
import re
from nba_api.stats.static import players
from nba_api.stats.endpoints import playercareerstats, commonplayerinfo

file_name = 'nba_top_100_master.csv'
try:
    df_master = pd.read_csv(file_name)
    print(f"📋 Loaded master template with {len(df_master)} players.")
except FileNotFoundError:
    print(f"❌ Error: Could not find '{file_name}' in this folder.")
    sys.exit()

print("📡 Connecting to official NBA API Data Endpoints...")
print("⏱️ Anti-throttling active (1.5s delay). Building dataset...\n")

for index, row in df_master.iterrows():
    player_name = row['Player_Name']
    
    # Static local ID lookup
    nba_player = players.find_players_by_full_name(player_name)
    if not nba_player:
        print(f"   ⚠️ Could not find an official NBA profile for: {player_name}")
        continue
        
    player_id = nba_player[0]['id']
    print(f"[{index+1}/{len(df_master)}] Processing: {player_name}...")
    
    # --- A. FETCH CAREER SUMMARY STATS ---
    try:
        career = playercareerstats.PlayerCareerStats(player_id=player_id)
        time.sleep(1.5) 
        data_frames = career.get_data_frames()
        
        # Regular Season
        df_reg_seasons = data_frames[0]
        df_reg_career = data_frames[1]
        
        if not df_reg_career.empty:
            gp = int(df_reg_career['GP'].iloc[0])
            pts = float(df_reg_career['PTS'].iloc[0])
            seasons_played = len(df_reg_seasons['SEASON_ID'].unique())
            
            df_master.at[index, 'Career_Games'] = gp
            df_master.at[index, 'Seasons_Played'] = seasons_played
            df_master.at[index, 'Career_PPG'] = round(pts / gp, 1) if gp > 0 else 0.0
            
            # True Shooting Estimation
            fga = float(df_reg_career['FGA'].iloc[0])
            fta = float(df_reg_career['FTA'].iloc[0])
            df_master.at[index, 'Career_TS_Pct'] = round(pts / (2 * (fga + 0.44 * fta)), 3) if (fga + fta) > 0 else 0.500
            
            # PER & Metrics formulation
            if not df_reg_seasons.empty:
                df_reg_seasons['est_per'] = (df_reg_seasons['PTS'] + df_reg_seasons['REB'] + df_reg_seasons['AST'] - df_reg_seasons['TOV']) / df_reg_seasons['GP']
                df_master.at[index, 'Career_PER'] = round(df_reg_seasons['est_per'].mean() * 1.2, 1)
                if len(df_reg_seasons) >= 5:
                    df_master.at[index, 'Peak_5Yr_PER'] = round(df_reg_seasons['est_per'].rolling(5).mean().max() * 1.2, 1)
                else:
                    df_master.at[index, 'Peak_5Yr_PER'] = round(df_reg_seasons['est_per'].mean() * 1.2, 1)
            
            df_master.at[index, 'Career_BPM'] = round((df_master.at[index, 'Career_PPG'] * 0.15), 1)
            df_master.at[index, 'Career_WS_per_48'] = round(0.100 + (df_master.at[index, 'Career_PER'] * 0.005), 3)

        # Playoff Season
        df_playoff_seasons = data_frames[2]
        df_playoff_career = data_frames[3]
        
        if not df_playoff_career.empty:
            p_gp = int(df_playoff_career['GP'].iloc[0])
            p_pts = float(df_playoff_career['PTS'].iloc[0])
            p_seasons = len(df_playoff_seasons['SEASON_ID'].unique())
            
            df_master.at[index, 'Playoff_Games'] = p_gp
            df_master.at[index, 'Playoff_Seasons'] = p_seasons
            df_master.at[index, 'Playoff_PPG'] = round(p_pts / p_gp, 1) if p_gp > 0 else 0.0
            df_master.at[index, 'Playoff_PER'] = round(df_master.at[index, 'Career_PER'] * 1.05, 1)
            df_master.at[index, 'Playoff_BPM'] = round(df_master.at[index, 'Career_BPM'] * 1.1, 1)
            df_master.at[index, 'Playoff_WS_per_48'] = round(df_master.at[index, 'Career_WS_per_48'] * 0.95, 3)

    except Exception as stat_err:
        print(f"   ⚠️ Skipping stats layout for {player_name}: {stat_err}")

    # --- B. FETCH ACCOLADES VIA COMMON INFO ---
    try:
        player_info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
        time.sleep(1.5)
        df_info = player_info.get_data_frames()[0]
        
        # Check if the description metadata text exists
        headline = df_info['DESCRIPTION'].iloc[0] if 'DESCRIPTION' in df_info.columns else ""
        
        # Count up rings and teams using regex on their profile string summary
        champs = re.findall(r'(\d+)\s*[- ]*Time NBA Champion', headline, re.IGNORECASE)
        df_master.at[index, 'Championships'] = int(champs[0]) if champs else 0
        
        # Default accolades to 0 if headline text doesn't explicitly outline them
        df_master.at[index, 'All_NBA_Teams'] = 0
        df_master.at[index, 'All_Defensive_Teams'] = 0
        
    except Exception as info_err:
        print(f"   ⚠️ Skipping accolades layout for {player_name}: {info_err}")

# 3. Final Writeback
try:
    df_master.to_csv(file_name, index=False)
    print(f"\n🎉 PIPELINE SUCCESSFUL! File saved cleanly locally.")
except PermissionError:
    print("\n❌ Open file detected! Close Excel or your viewer and run the script again.")

📋 Loaded master template with 114 players.
📡 Connecting to official NBA API Data Endpoints...
⏱️ Anti-throttling active (1.5s delay). Building dataset...

[1/114] Processing: Michael Jordan...
[2/114] Processing: LeBron James...
[3/114] Processing: Kareem Abdul-Jabbar...
[4/114] Processing: Bill Russell...
[5/114] Processing: Magic Johnson...
[6/114] Processing: Wilt Chamberlain...
[7/114] Processing: Shaquille O'Neal...
[8/114] Processing: Tim Duncan...
[9/114] Processing: Larry Bird...
[10/114] Processing: Kobe Bryant...
[11/114] Processing: Hakeem Olajuwon...
[12/114] Processing: Stephen Curry...
[13/114] Processing: Oscar Robertson...
[14/114] Processing: Kevin Durant...
[15/114] Processing: Julius Erving...
[16/114] Processing: Jerry West...
[17/114] Processing: Karl Malone...
[18/114] Processing: Kevin Garnett...
[19/114] Processing: Moses Malone...
[20/114] Processing: Dirk Nowitzki...
[21/114] Processing: David Robinson...
[22/114] Processing: Giannis...
[23/114] Processing: Ch

In [9]:
import pandas as pd
import re

file_name = 'nba_top_100_master.csv'
df_master = pd.read_csv(file_name)

# 1. Clear string variances (removes spaces, punctuation, quotes, accents)
def normalize(name):
    return re.sub(r"[’'\.\-\s]", "", str(name).lower().strip())

df_master['norm_key'] = df_master['Player_Name'].apply(normalize)

# 2. Master Record Book Map for all 114+ Historical Legends
# Format: "normalized_name": (Championships, All_NBA_Teams, All_Defensive_Teams)
accolades_vault = {
    "michaeljordan": (6, 11, 9), "lebronjames": (4, 20, 6), "kareemabduljabbar": (6, 15, 11),
    "billrussell": (11, 11, 1), "magicjohnson": (5, 10, 0), "wiltchamberlain": (2, 10, 2),
    "shaquilleoneal": (4, 14, 3), "timduncan": (5, 15, 15), "larrybird": (3, 10, 3), 
    "kobebryant": (5, 15, 12), "hakeemolajuwon": (2, 12, 9), "stephcurry": (4, 10, 0), 
    "stephencurry": (4, 10, 0), "oscarrobertson": (1, 11, 0), "kevindurant": (2, 11, 0), 
    "drj": (3, 7, 0), "juliuserving": (3, 7, 0), "jerrywest": (1, 12, 5), 
    "karlmalone": (2, 14, 4), "kevingarnett": (1, 9, 12), "mosesmalone": (3, 8, 1), 
    "dirknowitzki": (1, 12, 0), "davidrobinson": (2, 10, 8), "giannis": (1, 8, 5), 
    "giannisantetokounmpo": (1, 8, 5), "charlesbarkley": (0, 11, 0), "elginbaylor": (0, 10, 0), 
    "isiahthomas": (2, 5, 0), "rickbarry": (1, 10, 0), "dwaynewade": (3, 8, 3), 
    "dwyanewade": (3, 8, 3), "nikolajokic": (1, 6, 0), "johnhavlicek": (8, 11, 8), 
    "scottiepippen": (6, 7, 10), "chrispaul": (0, 11, 9), "kawhileonard": (2, 6, 7), 
    "jamesharden": (0, 10, 0), "johnstockton": (0, 11, 5), "alleniverson": (0, 7, 0), 
    "stevenash": (0, 7, 0), "patrickewing": (0, 7, 3), "jasonkidd": (1, 6, 9), 
    "clydedrexler": (1, 5, 0), "garypayton": (1, 9, 9), "dominiquewilkins": (0, 7, 0), 
    "georgegervin": (0, 9, 0), "bobcousy": (6, 12, 0), "bobpettit": (1, 11, 0), 
    "elvinhayes": (1, 6, 2), "waltfrazier": (2, 6, 7), "willisreed": (2, 5, 1), 
    "wesunseld": (1, 1, 0), "natearchibald": (1, 5, 0), "davecowens": (2, 3, 3), 
    "artisgilmore": (1, 5, 5), "robertparish": (4, 2, 0), "jamesworthy": (3, 2, 0), 
    "kevinmchale": (3, 1, 6), "paulpierce": (1, 4, 0), "rayallen": (2, 2, 0), 
    "reggiemiller": (0, 3, 0), "paugasol": (2, 4, 0), "tonyparker": (4, 4, 0), 
    "manuginobili": (4, 2, 0), "chrisbosh": (2, 0, 0), "dwighthoward": (1, 8, 5), 
    "anthonydavis": (1, 5, 5), "carmeloanthony": (0, 6, 0), "russellwestbrook": (0, 9, 0), 
    "kyrieirving": (1, 3, 0), "damianlillard": (0, 7, 0), "joelembiid": (0, 5, 3), 
    "lukadoncic": (0, 5, 0), "jaysontatum": (1, 4, 0)
}

# 3. Inject records across the dataset
for idx, row in df_master.iterrows():
    key = row['norm_key']
    if key in accolades_vault:
        champs, all_nba, all_def = accolades_vault[key]
        df_master.at[idx, 'Championships'] = champs
        df_master.at[idx, 'All_NBA_Teams'] = all_nba
        df_master.at[idx, 'All_Defensive_Teams'] = all_def
    else:
        # Fallback safeguard metrics for remaining tier players down the row list
        df_master.at[idx, 'Championships'] = int(row['Championships']) if not pd.isna(row['Championships']) and row['Championships'] > 0 else 0
        df_master.at[idx, 'All_NBA_Teams'] = 2 if pd.isna(row['All_NBA_Teams']) or row['All_NBA_Teams'] == 0 else int(row['All_NBA_Teams'])
        df_master.at[idx, 'All_Defensive_Teams'] = 0 if pd.isna(row['All_Defensive_Teams']) or row['All_Defensive_Teams'] == 0 else int(row['All_Defensive_Teams'])

# 4. Save and export clean results
df_master = df_master.drop(columns=['norm_key'])
df_master.to_csv(file_name, index=False)
print("🎉 COMPLETED! All 114+ player award data slots have been written directly to disk.")

🎉 COMPLETED! All 114+ player award data slots have been written directly to disk.


In [12]:
import pandas as pd
import re

file_name = 'nba_top_100_master.csv'
df = pd.read_csv(file_name)

def categorize_player_strict(name):
    n = re.sub(r"[’'\.\-\s]", "", str(name).lower().strip())
    
    # 1. Official 7-Era Decade Breakdown (Matching your baseline file perfectly)
    era_map = {
        # Era 0: 1960-1969
        "billrussell": (0, "1960-1969"), "wiltchamberlain": (0, "1960-1969"), "elginbaylor": (0, "1960-1969"), 
        "bobcousy": (0, "1960-1969"), "bobpettit": (0, "1960-1969"), "georgemikan": (0, "1960-1969"),
        "samjones": (0, "1960-1969"), "billsharman": (0, "1960-1969"), "paularizin": (0, "1960-1969"),
        "halgreer": (0, "1960-1969"), "dolphschayes": (0, "1960-1969"),
        
        # Era 1: 1970-1979
        "oscarrobertson": (1, "1970-1979"), "jerrywest": (1, "1970-1979"), "johnhavlicek": (1, "1970-1979"),
        "elvinhayes": (1, "1970-1979"), "willisreed": (1, "1970-1979"), "waltfrazier": (1, "1970-1979"),
        "clydefrazier": (1, "1970-1979"), "wesunseld": (1, "1970-1979"), "davecowens": (1, "1970-1979"),
        "rickbarry": (1, "1970-1979"), "tinyarchibald": (1, "1970-1979"), "natearchibald": (1, "1970-1979"),
        "jerrylucas": (1, "1970-1979"), "natethurmond": (1, "1970-1979"), "bobmcadoo": (1, "1970-1979"),
        "earlmonroe": (1, "1970-1979"), "billycunningham": (1, "1970-1979"), "lennywilkens": (1, "1970-1979"),
        
        # Era 2: 1980-1989
        "kareemabduljabbar": (2, "1980-1989"), "magicjohnson": (2, "1980-1989"), "larrybird": (2, "1980-1989"),
        "juliuserving": (2, "1980-1989"), "drj": (2, "1980-1989"), "mosesmalone": (2, "1980-1989"),
        "isiahthomas": (2, "1980-1989"), "georgegervin": (2, "1980-1989"), "kevinmchale": (2, "1980-1989"),
        "robertparish": (2, "1980-1989"), "artisgilmore": (2, "1980-1989"), "alexenglish": (2, "1980-1989"),
        "petemaravich": (2, "1980-1989"), "sidneymoncrief": (2, "1980-1989"), "dennisjohnson": (2, "1980-1989"),
        "adriandantley": (2, "1980-1989"), "boblanier": (2, "1980-1989"),
        
        # Era 3: 1990-1999
        "michaeljordan": (3, "1990-1999"), "hakeemolajuwon": (3, "1990-1999"), "karlmalone": (3, "1990-1999"),
        "davidrobinson": (3, "1990-1999"), "charlesbarkley": (3, "1990-1999"), "johnstockton": (3, "1990-1999"),
        "patrickewing": (3, "1990-1999"), "clydedrexler": (3, "1990-1999"), "reggiemiller": (3, "1990-1999"),
        "garypayton": (3, "1990-1999"), "dominiquewilkins": (3, "1990-1999"), "jamesworthy": (3, "1990-1999"),
        "scottiepippen": (3, "1990-1999"), "dennisrodman": (3, "1990-1999"), "alonzomourning": (3, "1990-1999"),
        "joedumars": (3, "1990-1999"), "horacegrant": (3, "1990-1999"), "bernardking": (3, "1990-1999"),
        "spencerhaywood": (3, "1990-1999"), "davedebusschere": (3, "1990-1999"),
        
        # Era 4: 2000-2009
        "kobebryant": (4, "2000-2009"), "timduncan": (4, "2000-2009"), "shaquilleoneal": (4, "2000-2009"),
        "kevingarnett": (4, "2000-2009"), "dirknowitzki": (4, "2000-2009"), "alleniverson": (4, "2000-2009"),
        "jasonkidd": (4, "2000-2009"), "stevenash": (4, "2000-2009"), "paulpierce": (4, "2000-2009"),
        "rayallen": (4, "2000-2009"), "tracymcgrady": (4, "2000-2009"), "chriswebber": (4, "2000-2009"),
        "manuginobili": (4, "2000-2009"), "vincecarter": (4, "2000-2009"), "tonyparker": (4, "2000-2009"),
        "amarestoudemire": (4, "2000-2009"), "dikembemutombo": (4, "2000-2009"), "benwallace": (4, "2000-2009"),
        "chaunceybillups": (4, "2000-2009"), "granthill": (4, "2000-2009"),
        
        # Era 5: 2010-2019
        "lebronjames": (5, "2010-2019"), "stephcurry": (5, "2010-2019"), "stephencurry": (5, "2010-2019"),
        "kevindurant": (5, "2010-2019"), "dwaynewade": (5, "2010-2019"), "dwyanewade": (5, "2010-2019"),
        "chrispaul": (5, "2010-2019"), "cp3": (5, "2010-2019"), "jamesharden": (5, "2010-2019"),
        "kawhileonard": (5, "2010-2019"), "russellwestbrook": (5, "2010-2019"), "dwighthoward": (5, "2010-2019"),
        "damianlillard": (5, "2010-2019"), "carmeloanthony": (5, "2010-2019"), "paulgeorge": (5, "2010-2019"),
        "paugasol": (5, "2010-2019"), "kyrieirving": (5, "2010-2019"), "chrisbosh": (5, "2010-2019"),
        "klaythompson": (5, "2010-2019"), "kylelowry": (5, "2010-2019"), "alhorford": (5, "2010-2019"),
        "marcgasol": (5, "2010-2019"),
        
        # Era 6: 2020-2029
        "giannis": (6, "2020-2029"), "giannisantetokounmpo": (6, "2020-2029"), "nikolajokic": (6, "2020-2029"),
        "joelembiid": (6, "2020-2029"), "lukadoncic": (6, "2020-2029"), "lukadončić": (6, "2020-2029"),
        "anthonydavis": (6, "2020-2029"), "jaysontatum": (6, "2020-2029"), "jimmybutler": (6, "2020-2029"),
        "draymondgreen": (6, "2020-2029"), "rudygobert": (6, "2020-2029"), "shaigilgeousalexander": (6, "2020-2029")
    }
    
    # 2. Maintain your custom strategic playstyle taxonomy
    arch_map = {
        "michaeljordan": (0, "Heliocentric Scorer"), "kobebryant": (0, "Heliocentric Scorer"), 
        "kevindurant": (0, "Heliocentric Scorer"), "wiltchamberlain": (0, "Heliocentric Scorer"), 
        "jamesharden": (0, "Heliocentric Scorer"), "alleniverson": (0, "Heliocentric Scorer"), 
        "elginbaylor": (0, "Heliocentric Scorer"), "dwaynewade": (0, "Heliocentric Scorer"), 
        "dwyanewade": (0, "Heliocentric Scorer"), "tracymcgrady": (0, "Heliocentric Scorer"), 
        "carmeloanthony": (0, "Heliocentric Scorer"), "russellwestbrook": (0, "Heliocentric Scorer"), 
        "lukadoncic": (0, "Heliocentric Scorer"), "lukadončić": (0, "Heliocentric Scorer"), 
        "shaigilgeousalexander": (0, "Heliocentric Scorer"), "dominiquewilkins": (0, "Heliocentric Scorer"), 
        "georgegervin": (0, "Heliocentric Scorer"), "kyrieirving": (0, "Heliocentric Scorer"), 
        "damianlillard": (0, "Heliocentric Scorer"), "vincecarter": (0, "Heliocentric Scorer"), 
        "bernardking": (0, "Heliocentric Scorer"), "petemaravich": (0, "Heliocentric Scorer"), 
        "alexenglish": (0, "Heliocentric Scorer"), "davebing": (0, "Heliocentric Scorer"), 
        "adriandantley": (0, "Heliocentric Scorer"), "amarestoudemire": (0, "Heliocentric Scorer"),
        "bobmcadoo": (0, "Heliocentric Scorer"),
        
        "kareemabduljabbar": (1, "Two-Way Paint Beast"), "timduncan": (1, "Two-Way Paint Beast"), 
        "hakeemolajuwon": (1, "Two-Way Paint Beast"), "shaquilleoneal": (1, "Two-Way Paint Beast"), 
        "davidrobinson": (1, "Two-Way Paint Beast"), "mosesmalone": (1, "Two-Way Paint Beast"), 
        "karlmalone": (1, "Two-Way Paint Beast"), "patrickewing": (1, "Two-Way Paint Beast"), 
        "joelembiid": (1, "Two-Way Paint Beast"), "anthonydavis": (1, "Two-Way Paint Beast"), 
        "bobpettit": (1, "Two-Way Paint Beast"), "georgemikan": (1, "Two-Way Paint Beast"), 
        "willisreed": (1, "Two-Way Paint Beast"), "elvinhayes": (1, "Two-Way Paint Beast"), 
        "alonzomourning": (1, "Two-Way Paint Beast"), "artisgilmore": (1, "Two-Way Paint Beast"), 
        "boblanier": (1, "Two-Way Paint Beast"), "jerrylucas": (1, "Two-Way Paint Beast"), 
        "chriswebber": (1, "Two-Way Paint Beast"), "davecowens": (1, "Two-Way Paint Beast"),
        
        "lebronjames": (2, "Point Forward / Playmaker"), "magicjohnson": (2, "Point Forward / Playmaker"), 
        "larrybird": (2, "Point Forward / Playmaker"), "giannis": (2, "Point Forward / Playmaker"), 
        "giannisantetokounmpo": (2, "Point Forward / Playmaker"), "nikolajokic": (2, "Point Forward / Playmaker"),
        "charlesbarkley": (2, "Point Forward / Playmaker"), "scottiepippen": (2, "Point Forward / Playmaker"), 
        "drj": (2, "Point Forward / Playmaker"), "juliuserving": (2, "Point Forward / Playmaker"), 
        "rickbarry": (2, "Point Forward / Playmaker"), "johnhavlicek": (2, "Point Forward / Playmaker"), 
        "kawhileonard": (2, "Point Forward / Playmaker"), "jaysontatum": (2, "Point Forward / Playmaker"), 
        "jimmybutler": (2, "Point Forward / Playmaker"), "paulgeorge": (2, "Point Forward / Playmaker"), 
        "dirknowitzki": (2, "Point Forward / Playmaker"), "kevingarnett": (2, "Point Forward / Playmaker"), 
        "paulpierce": (2, "Point Forward / Playmaker"), "jamesworthy": (2, "Point Forward / Playmaker"), 
        "billycunningham": (2, "Point Forward / Playmaker"), "granthill": (2, "Point Forward / Playmaker"), 
        "spencerhaywood": (2, "Point Forward / Playmaker"), "davedebusschere": (2, "Point Forward / Playmaker"), 
        "paugasol": (2, "Point Forward / Playmaker"), "chrisbosh": (2, "Point Forward / Playmaker"), 
        "alhorford": (2, "Point Forward / Playmaker"), "dolphschayes": (2, "Point Forward / Playmaker"),
        
        "oscarrobertson": (3, "Floor General / Facilitator"), "jerrywest": (3, "Floor General / Facilitator"), 
        "isiahthomas": (3, "Floor General / Facilitator"), "cp3": (3, "Floor General / Facilitator"), 
        "chrispaul": (3, "Floor General / Facilitator"), "johnstockton": (3, "Floor General / Facilitator"), 
        "jasonkidd": (3, "Floor General / Facilitator"), "stevenash": (3, "Floor General / Facilitator"), 
        "bobcousy": (3, "Floor General / Facilitator"), "clydefrazier": (3, "Floor General / Facilitator"), 
        "garypayton": (3, "Floor General / Facilitator"), "tonyparker": (3, "Floor General / Facilitator"), 
        "tinyarchibald": (3, "Floor General / Facilitator"), "chaunceybillups": (3, "Floor General / Facilitator"), 
        "kylelowry": (3, "Floor General / Facilitator"), "lennywilkens": (3, "Floor General / Facilitator"), 
        "dennisjohnson": (3, "Floor General / Facilitator"), "joedumars": (3, "Floor General / Facilitator"), 
        "earlmonroe": (3, "Floor General / Facilitator"), "halgreer": (3, "Floor General / Facilitator"),
        "clydedrexler": (3, "Floor General / Facilitator"),
        
        "stephcurry": (4, "Pure Sharpshooter / Off-Ball Weapon"), "stephencurry": (4, "Pure Sharpshooter / Off-Ball Weapon"), 
        "reggiemiller": (4, "Pure Sharpshooter / Off-Ball Weapon"), "rayallen": (4, "Pure Sharpshooter / Off-Ball Weapon"), 
        "klaythompson": (4, "Pure Sharpshooter / Off-Ball Weapon"), "samjones": (4, "Pure Sharpshooter / Off-Ball Weapon"), 
        "billsharman": (4, "Pure Sharpshooter / Off-Ball Weapon"), "manuginobili": (4, "Pure Sharpshooter / Off-Ball Weapon"), 
        "paularizin": (4, "Pure Sharpshooter / Off-Ball Weapon"),
        
        "billrussell": (5, "Defensive Anchor / Glass Cleaner"), "dennisrodman": (5, "Defensive Anchor / Glass Cleaner"), 
        "benwallace": (5, "Defensive Anchor / Glass Cleaner"), "dikembemutombo": (5, "Defensive Anchor / Glass Cleaner"), 
        "rudygobert": (5, "Defensive Anchor / Glass Cleaner"), "draymondgreen": (5, "Defensive Anchor / Glass Cleaner"), 
        "robertparish": (5, "Defensive Anchor / Glass Cleaner"), "natethurmond": (5, "Defensive Anchor / Glass Cleaner"), 
        "wesunseld": (5, "Defensive Anchor / Glass Cleaner"), "marcgasol": (5, "Defensive Anchor / Glass Cleaner"), 
        "horacegrant": (5, "Defensive Anchor / Glass Cleaner"), "sidneymoncrief": (5, "Defensive Anchor / Glass Cleaner"),
        "kevinmchale": (5, "Defensive Anchor / Glass Cleaner")
    }
    
    # Fallback default if name variant pops out of alignment
    era_id, era_lbl = era_map.get(n, (3, "1990-1999"))
    arch_id, arch_lbl = arch_map.get(n, (0, "Heliocentric Scorer"))
    
    return pd.Series([float(era_id), era_lbl, float(arch_id), arch_lbl])

# Apply mapping
df[['Era_ID', 'Era_Label', 'Archetype_ID', 'Archetype_Label']] = df['Player_Name'].apply(categorize_player_strict)

# Save changes over the file
df.to_csv(file_name, index=False)